In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change this if your project folder is somewhere else in Drive.
%cd "/content/drive/MyDrive/Final Project/solver"

In [ ]:
%ls

In [ ]:
%pip install -q -r requirements-sft.txt
%pip install -q -U huggingface_hub

In [ ]:
import torch
import transformers
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## Hugging Face Login

Run the next cell and paste a Hugging Face token into the widget. Do not store tokens directly in this notebook.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Base Model Smoke Test

This checks that `HuggingFaceTB/SmolLM2-135M-Instruct` loads and can generate before training.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
prompt = "What is the derivative of x^2?"

messages = [
    {"role": "system", "content": "You are a careful mathematical problem solver. Provide a complete solution with all necessary reasoning."},
    {"role": "user", "content": f"Solve the following problem:\n\n{prompt}"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
# Free VRAM from the smoke test before training.
del model
import gc
gc.collect()
torch.cuda.empty_cache()

## Stage A: Preprocess Raw Dataset

In [ ]:
!python prepare_solver_sft_data.py \
  --input solver_full_trajectory_dataset.jsonl \
  --output-dir .

## Stage B: Check Chat Formatting And Token Lengths

In [ ]:
!python format_dataset.py \
  --model HuggingFaceTB/SmolLM2-135M-Instruct \
  --files train_sft.jsonl val_sft.jsonl test_sft.jsonl \
  --max-seq-length 2048

## Stage C: Train SmolLM2-135M Solver

This command uses full fine-tuning, not QLoRA. If Colab runs out of memory, reduce `--per-device-train-batch-size` to `2` and increase `--gradient-accumulation-steps` to `8`.

In [ ]:
!accelerate launch \
  --num_processes 1 \
  --num_machines 1 \
  --mixed_precision fp16 \
  --dynamo_backend no \
  train_solver_sft.py \
  --model HuggingFaceTB/SmolLM2-135M-Instruct \
  --train-file train_sft.jsonl \
  --val-file val_sft.jsonl \
  --output-dir outputs/smollm2-135m-solver-sft \
  --max-seq-length 2048 \
  --per-device-train-batch-size 4 \
  --per-device-eval-batch-size 4 \
  --gradient-accumulation-steps 4 \
  --learning-rate 5e-5 \
  --epochs 5

## Stage D: Run The Trained Solver

In [ ]:
!python run_solver.py \
  --model outputs/smollm2-135m-solver-sft \
  --problem "Compute 55 times 1212 minus 15 times 1212."

## Stage E: Quick Evaluation

In [ ]:
!python eval_solver.py \
  --model outputs/smollm2-135m-solver-sft \
  --test-file test_sft.jsonl \
  --output-file eval_predictions.jsonl \
  --max-examples 5